# Causal Self-Attention

In autoregressive language models, tokens must only attend to preceding tokens to prevent information leakage from future positions during generation. This notebook implements causal self-attention by applying an upper-triangular mask of negative infinity values to the raw attention scores before the softmax operation.

## Table of Contents
1. Imports
2. Causal Self-Attention Implementation
3. Verification and Shape Testing


## 1. Imports

Imports PyTorch neural network modules and functional operations required for masked attention.


In [1]:
import torch 
import math 
import torch.nn as nn 
import torch.nn.functional as F

## 2. Causal Self-Attention Implementation

Implements `CausalAttention`. The attention score matrix $S = Q K^T / \sqrt{d_k}$ has shape `(B, T, T)`. An upper-triangular mask fills all positions where $j > i$ with $-\infty$, ensuring that $\text{softmax}(S)_{ij} = 0$ for all future tokens $j$. Dropout is applied to the resulting attention weights for regularization.


In [9]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)

        d_k = key.size(-1)
        atten_score = query @ key.transpose(-2, -1)

        seq_len = atten_score.size(-1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()

        masked_atten_score = atten_score.masked_fill(~mask, float('-inf'))

        atten_weights = torch.softmax(masked_atten_score / math.sqrt(d_k), dim=-1)

        torch.manual_seed(789)
        drop_atten_weights = F.dropout(atten_weights, p=0.1)

        context = drop_atten_weights @ value
        return context

## 3. Verification and Shape Testing

Tests `CausalAttention` with input dimension $d_{in}=4$ and output dimension $d_{out}=2$. Verifies that feeding an input of shape `(1, 3, 4)` preserves causal ordering and yields output tensor shape `(1, 3, 2)`.


In [10]:
d_in = 4
d_out = 2
torch.manual_seed(789)

inputs = torch.randn(1, 3, d_in)

sa = CausalAttention(d_in, d_out)
print(sa(inputs))

tensor([[[-0.4684,  0.0188],
         [-0.3012, -0.2176],
         [-0.1737, -0.1098]]], grad_fn=<UnsafeViewBackward0>)
